In [ ]:
%%configure
{ "conf": { "spark.fabric.resourceProfile": "readHeavyForPBI" } }

# The paper's Direct Lake on OneLake arm

The same rows as the Databricks arms -- read out of the mirrored Unity Catalog schema
`tpcds_sf{sf}_layout` and rewritten by Fabric Spark exactly as *Modern Power BI Architecture
Choices for Reporting on Azure Databricks* built its Fabric arm.

**Section 5.2**, all of it:

| | how |
|---|---|
| V-Order enabled | `readHeavyForPBI` (cell 1) |
| Optimize Write enabled | `readHeavyForPBI` |
| Optimize Write target file size 1 GB | `readHeavyForPBI` -- the paper took the 1 GB from this profile |
| ZSTD compression | set below; the profile does not set it |
| OPTIMIZE, VACUUM, ANALYZE on all tables | the maintenance cell |
| fact tables partitioned, each partition sorted | the write cell + the maintenance cell |

**Table 4.6.1**, Direct Lake on OneLake row:

| | partitioned by | sorted within partition on |
|---|---|---|
| `store_sales` | `ss_sold_date_sk` | `ss_addr_sk` |
| `catalog_sales` | `cs_sold_date_sk` | `cs_bill_addr_sk` |

The only parameter is `sf`.

In [ ]:
sf = 100
# "paper"    = the paper's own arm: V-Order + Optimize Write + ZSTD, facts partitioned by the date
#              key and sorted within each partition, then OPTIMIZE / VACUUM / ANALYZE. Schema tpcds_sf{sf}.
# "vonly"    = V-ORDER AND NOTHING ELSE: no partitionBy, no sort, no OPTIMIZE. Schema
#              tpcds_sf{sf}_vonly.
#
# Why "vonly" exists: the paper's arm changes four things at once, and its partitionBy lands
# ONE FILE PER DATE PARTITION -- 1,823 files at every scale factor, so ~143k-row row groups. Direct
# Lake segments ARE row groups, so that arm gets ~1,800 segments where a 6M-row Databricks arm gets
# ~46. Any advantage it shows could be V-Order, or could be that partition. `vonly` separates them:
# same encoding, same compression, same writer, no partition.
#
variant = "paper"
src_schema = ""      # empty -> tpcds_sf{sf}_default in the mirrored catalog

In [ ]:
import time

import notebookutils

MIRROR = "01b539f3-4a9d-45ef-b1ef-0ba59552eb21"   # Mirrored Azure Databricks catalog item

sf     = int(sf)
assert variant in ("paper", "vonly"), \
    f"variant must be 'paper' or 'vonly', not {variant!r}"
# The source used to be tpcds_sf{sf}_layout, which no longer exists. It is the UNSORTED arm now, on
# purpose: every Fabric arm must be written from the same rows in the same order, and the unsorted
# arm is the one that imposes no ordering of its own on what Fabric Spark then writes.
SRC_SCHEMA = src_schema or f"tpcds_sf{sf}_default"
SRC    = (f"abfss://51650f82-6bb5-4023-b0ab-db197d32e0be@onelake.dfs.fabric.microsoft.com/"
          f"{MIRROR}/Tables/{SRC_SCHEMA}")
# One schema per variant. Writing two into tpcds_sf{sf} would leave two layouts wearing one name,
# which is the whole failure mode this project is built to avoid.
SCHEMA = {"paper": f"tpcds_sf{sf}", "vonly": f"tpcds_sf{sf}_vonly"}[variant]

# Table 4.6.1: table -> (partition column, within-partition sort column). The first is the date
# key -- the key the Databricks cluster arm uses and the one 23 of the paper's 24 captured
# queries filter on.
FACTS  = {"store_sales":   ("ss_sold_date_sk", "ss_addr_sk"),
          "catalog_sales": ("cs_sold_date_sk", "cs_bill_addr_sk")}
DIMS   = ["catalog_page", "customer_address", "customer_demographics", "date_dim",
          "item", "promotion", "ship_mode", "store"]
TABLES = DIMS + list(FACTS)

# The target is the ATTACHED lakehouse, so every statement below is ordinary SQL against
# `{SCHEMA}.{table}`. If the attachment is ever lost, saveAsTable would land somewhere else --
# fail here instead.
assert notebookutils.runtime.context.get("defaultLakehouseId") == "f19d8f93-2956-4cf7-a1bb-c526d0a953cd", \
    "this notebook must run with the tpcds_vorder lakehouse attached as its default"

# The one section 5.2 item readHeavyForPBI does not set. Applied here rather than in %%configure
# because the resource profile is applied AFTER session conf.
spark.conf.set("spark.sql.parquet.compression.codec", "zstd")

# Neither arm here declares clustering keys, but the pin stays: liquid clustering turns
# v2Checkpoint on by default and Direct Lake / VertiPaq cannot read a v2 checkpoint AT ALL --
# the table mirrors and then fails to load. Same pin as the Databricks arm.
spark.conf.set("spark.databricks.delta.properties.defaults.checkpointPolicy", "classic")

for k in ("spark.fabric.resourceProfile",
          "spark.sql.parquet.vorder.default",
          "spark.sql.parquet.compression.codec",
          "spark.microsoft.delta.optimizeWrite.enabled",
          "spark.microsoft.delta.optimizeWrite.binSize",
          "spark.databricks.delta.properties.defaults.checkpointPolicy"):
    print(f"{k} = {spark.conf.get(k, '<unset>')}")

# The profile is what turns V-Order on. If it did not take, this arm is not a V-Order arm.
assert spark.conf.get("spark.sql.parquet.vorder.default", "false").lower() == "true", \
    "V-Order is OFF: the readHeavyForPBI resource profile did not take"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
print(f"\nSF{sf}: {SRC}\n   ->  tpcds_vorder.{SCHEMA}   (variant={variant})")

In [ ]:
# NEVER OVERWRITES. A table that already exists is left exactly as it is and the loop moves on.
# Re-running this notebook is therefore free and safe: it finishes a partial build instead of
# redoing an hour of writes, which is what `mode("overwrite")` used to do on every re-run --
# silently, and with a maintenance cell after it that might then fail again.
#
# The `overwrite` on the writes below is what CREATES the table on a first run; it is only ever
# reached for a table that does not exist yet, so it can never destroy a built arm.
#
# To genuinely rebuild, DROP the schema by hand first. That is deliberate: same rule as the
# Databricks side, where the arm lives in the schema suffix and an existing table stops the run.
try:
    _existing = {r.tableName if hasattr(r, "tableName") else r[0]
                 for r in spark.sql(f"SHOW TABLES IN {SCHEMA}").collect()}
except Exception:                                   # noqa: BLE001 -- schema not created yet
    _existing = set()
if _existing:
    print(f"{SCHEMA}: {len(_existing)} table(s) already present, they will be SKIPPED "
          f"-- drop the schema by hand to force a rebuild", flush=True)

_written, _skipped = [], []
for t in TABLES:
    t0 = time.time()
    if t in _existing:
        _skipped.append(t)
        print(f"{t}: exists, skipped", flush=True)
        continue
    src = spark.read.format("delta").load(f"{SRC}/{t}")

    w = (src.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .option("delta.parquet.vorder.enabled", "true"))
    # The ONLY difference between paper and vonly. `vonly` writes the facts exactly as it writes
    # the dimensions -- V-Order, Optimize Write, ZSTD, and nothing that reshapes the files.
    if variant == "paper" and t in FACTS:
        w = w.partitionBy(FACTS[t][0])
    w.saveAsTable(f"{SCHEMA}.{t}")
    _written.append(t)
    print(f"{t}: written in {time.time() - t0:,.0f}s", flush=True)

print()
print(f"{len(_written)} written, {len(_skipped)} skipped, {len(TABLES)} expected")
assert len(_written) + len(_skipped) == len(TABLES), "a table is neither written nor skipped"


In [ ]:
# Section 4.6 and 5.2: OPTIMIZE, VACUUM and ANALYZE on all tables, ZORDER on the facts.
# `vonly` skips all of it: the sort and the OPTIMIZE both rewrite the files, so running either
# would put back the thing this variant exists to remove.
if variant == "paper":
    for t in TABLES:
        t0 = time.time()
        z = f" ZORDER BY ({FACTS[t][1]})" if t in FACTS else ""
        spark.sql(f"OPTIMIZE {SCHEMA}.{t}{z}")
        spark.sql(f"VACUUM {SCHEMA}.{t} RETAIN 168 HOURS")
        spark.sql(f"ANALYZE TABLE {SCHEMA}.{t} COMPUTE STATISTICS FOR ALL COLUMNS")
        print(f"{t}: optimize{z} + vacuum + analyze in {time.time() - t0:,.0f}s", flush=True)
else:
    print("vonly: no OPTIMIZE, no sort, no VACUUM -- V-Order and the write, nothing else")

## Report

Rows, file count and size per table. File count and GB are what the paper's Table 4.6.2 reports
for this arm: 1,823 files / 1,836 files for `store_sales` / `catalog_sales` at every scale factor,
which is one file per `*_sold_date_sk` partition.

In [ ]:
import pandas as pd

rows = []
for t in TABLES:
    d = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{t}").collect()[0].asDict()
    props = d.get("properties") or {}
    rows.append({"table": t,
                 "rows": spark.table(f"{SCHEMA}.{t}").count(),
                 "files": d["numFiles"],
                 "GB": round(d["sizeInBytes"] / 1024 ** 3, 2),
                 "partitioned_by": ",".join(d.get("partitionColumns") or []),
                 "clustered_by": ",".join(d.get("clusteringColumns") or []),
                 "checkpoint": props.get("delta.checkpointPolicy"),
                 "vorder": props.get("delta.parquet.vorder.enabled")})
display(pd.DataFrame(rows))